<a href="https://colab.research.google.com/github/ameesha543/Statistical-Learning-e23095/blob/main/Assignment_5/Question_1/Part-B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np

class DiagnosticsLayer:
    """
    A class to compute Hotelling's T^2 and Q statistics for real-time diagnostics.
    """
    def __init__(self, baseline_archive):
        """
        Initializes the DiagnosticsLayer with a baseline archive.

        Args:
            baseline_archive (np.ndarray): An n x m array of nominal snapshots,
                                           where n is the number of samples and m is the number of sensors.
        """
        if not isinstance(baseline_archive, np.ndarray):
            raise TypeError("baseline_archive must be a NumPy array.")
        if baseline_archive.ndim != 2:
            raise ValueError("baseline_archive must be a 2D array.")

        self.baseline_archive = baseline_archive
        self.n_baseline, self.m_sensors = baseline_archive.shape

        # Calculate mean from baseline for centering incoming data
        self.baseline_mean = np.mean(baseline_archive, axis=0)

        # Center the baseline archive for covariance calculation
        centered_baseline = baseline_archive - self.baseline_mean

        # Compute unbiased sample covariance matrix S
        # S = (X^T X) / (n-1) where X is the centered data
        self.S = (centered_baseline.T @ centered_baseline) / (self.n_baseline - 1)

        # Decompose S using np.linalg.eigh (returns eigenvalues and eigenvectors)
        eigenvalues, eigenvectors = np.linalg.eigh(self.S)

        # Sort eigenvalues in descending order and reorder eigenvectors accordingly
        # np.argsort returns indices that would sort an array, [::-1] reverses it for descending.
        idx = eigenvalues.argsort()[::-1]
        self.eigenvalues = eigenvalues[idx]
        self.eigenvectors = eigenvectors[:, idx] # Eigenvectors are columns of the matrix

    def compute_metrics(self, data):
        """
        Computes Hotelling's T^2 and Q statistics for incoming data.

        Args:
            data (np.ndarray): An N x m array of new realizations to analyze,
                               where N is the number of new samples and m is the number of sensors.

        Returns:
            dict: A dictionary where keys are 'k=1', 'k=2', 'k=3' and values are dictionaries
                  containing 'mean_T2' and 'mean_Q' for that k.
        """
        if not isinstance(data, np.ndarray):
            raise TypeError("Input data must be a NumPy array.")
        if data.ndim != 2 or data.shape[1] != self.m_sensors:
            raise ValueError(f"Input data must be a 2D array with {self.m_sensors} columns (sensors).")

        # 1. Standardize and center the incoming realizations (using baseline mean)
        # For Hotelling's T^2 and Q, data is centered using the mean of the nominal state.
        centered_incoming_data = data - self.baseline_mean

        # Project the centered data onto the principal components
        # z_scores = centered_incoming_data @ eigenvectors
        z_scores = centered_incoming_data @ self.eigenvectors

        m = self.m_sensors # Number of sensors/dimensions

        k_values = [1, 2, 3] # Possible subspace cutoff values
        results = {}

        for k in k_values:
            T2_values_for_k = []
            Q_values_for_k = []

            # Loop through each realization (row) in the incoming data
            for i in range(z_scores.shape[0]):
                current_z = z_scores[i, :]

                # Calculate Hotelling’s T^2_i = sum(z_j^2 / lambda_j) for j=1 to k
                # Use only the first k principal components and their corresponding eigenvalues
                T2_i = np.sum((current_z[:k]**2) / self.eigenvalues[:k])
                T2_values_for_k.append(T2_i)

                # Calculate Q_i (residual) = sum(z_j^2) for j=k+1 to m
                # Use principal components from k+1 to m
                if k < m:
                    Q_i = np.sum(current_z[k:m]**2)
                else:
                    Q_i = 0 # If k >= m, all components are in T2, so Q is 0
                Q_values_for_k.append(Q_i)

            # Compute the expected value (empirical mean) of T^2 and Q across all samples for each k
            mean_T2_k = np.mean(T2_values_for_k)
            mean_Q_k = np.mean(Q_values_for_k)

            results[f'k={k}'] = {'mean_T2': mean_T2_k, 'mean_Q': mean_Q_k}

        return results

### Explanation of the `DiagnosticsLayer` Class

The `DiagnosticsLayer` class is designed to analyze structural component sensor data using two key multivariate statistical process control metrics: Hotelling's $T^2$ statistic and the residual $Q$ statistic. These metrics are crucial for identifying deviations from a nominal (healthy) state.

#### `__init__(self, baseline_archive)`
-   **Purpose:** Initializes the diagnostic system with a `baseline_archive` of nominal sensor readings.
-   **Process:**
    1.  **Stores Baseline:** Keeps a reference to the `baseline_archive` to compute statistical properties.
    2.  **Calculates Mean:** Computes the mean of each sensor reading from the `baseline_archive`. This mean is used to center both the baseline data (for covariance calculation) and any incoming data (for projection).
    3.  **Computes Covariance Matrix (S):** Calculates the unbiased sample covariance matrix $\mathbf{S}$ from the centered `baseline_archive`. This matrix describes the variances and covariances among the sensor readings in the nominal state.
    4.  **Eigen-decomposition:** Performs an eigen-decomposition of $\mathbf{S}$ using `np.linalg.eigh`. This yields eigenvalues ($\widehat{\lambda}$) and eigenvectors (principal components).
    5.  **Sorts Components:** Sorts the eigenvalues in descending order and reorders the corresponding eigenvectors. The eigenvalues represent the variance captured by each principal component, and sorting ensures that the most significant components (those capturing the most variance) come first.

#### `compute_metrics(self, data)`
-   **Purpose:** Analyzes new incoming sensor `data` against the established baseline to calculate Hotelling's $T^2$ and the $Q$ statistics for different subspace truncation values ($k$).
-   **Process:**
    1.  **Centers Incoming Data:** Each sample in the `data` is centered by subtracting the `baseline_mean` (calculated during initialization). This ensures consistency with the centered data used for covariance estimation.
    2.  **Projects Data (z-scores):** The centered `data` is then projected onto the principal components (eigenvectors). The resulting `z_scores` represent the coordinates of each data point in the principal component space.
    3.  **Iterates through `k`:** Loops through predefined truncation values $k \in \{1, 2, 3\}$. For each $k$:
        -   **Hotelling's $T^2$ Calculation:** For each sample, it calculates $T^2_i = \sum_{j=1}^k \frac{z_{i,j}^2}{\widehat{\lambda}_j}$. This statistic measures the variation within the principal component subspace defined by the first $k$ components. It's sensitive to shifts in the mean of the process.
        -   **$Q$ Statistic Calculation:** For each sample, it calculates $Q_i = \sum_{j=k+1}^m z_{i,j}^2$. This statistic measures the residual variation in the subspace orthogonal to the first $k$ components. It's sensitive to changes in the process variance or relationships between variables not captured by the first $k$ principal components.
        -   **Computes Means:** The empirical mean of $T^2$ and $Q$ across all incoming samples is calculated for the current $k$.
    4.  **Returns Results:** A dictionary containing the mean $T^2$ and mean $Q$ values for each $k$ is returned.

### Demonstration of `DiagnosticsLayer`

Let's create a synthetic `baseline_archive` and some `new_realizations` to demonstrate the usage of the `DiagnosticsLayer` class and its `compute_metrics` method. We'll simulate `m=4` sensors and `n=3000` nominal snapshots for the baseline, and then some `N=100` new realizations.

In [2]:
# Define parameters for demonstration
n_baseline_samples = 3000  # n
m_sensors = 4            # m
n_new_realizations = 100

# Create a synthetic baseline archive (nominal data)
# We'll use a multivariate normal distribution for simplicity
np.random.seed(42) # for reproducibility
baseline_mean = np.array([10, 20, 15, 25])
baseline_cov = np.array([
    [1.0, 0.5, 0.2, 0.1],
    [0.5, 1.5, 0.3, 0.2],
    [0.2, 0.3, 1.2, 0.4],
    [0.1, 0.2, 0.4, 1.3]
])

baseline_archive = np.random.multivariate_normal(baseline_mean, baseline_cov, n_baseline_samples)

print(f"Baseline archive shape: {baseline_archive.shape}")

# Create an instance of the DiagnosticsLayer
diagnostic_system = DiagnosticsLayer(baseline_archive)

# Simulate new realizations
# Case 1: Nominal new data (should have low T2 and Q)
new_realizations_nominal = np.random.multivariate_normal(baseline_mean, baseline_cov, n_new_realizations)

# Case 2: Shifted mean (should result in higher T2)
shifted_mean = baseline_mean + np.array([2, 0, 0, 0]) # Shift only the first sensor's mean
new_realizations_shifted_mean = np.random.multivariate_normal(shifted_mean, baseline_cov, n_new_realizations)

# Case 3: Increased variance (should result in higher Q if not captured by first few PCs)
increased_cov = baseline_cov * 2 # Double the covariance
new_realizations_increased_variance = np.random.multivariate_normal(baseline_mean, increased_cov, n_new_realizations)

print(f"New realizations (nominal) shape: {new_realizations_nominal.shape}")

# Compute metrics for nominal new data
nominal_metrics = diagnostic_system.compute_metrics(new_realizations_nominal)
print("\nMetrics for Nominal New Realizations:")
for k_val, metrics in nominal_metrics.items():
    print(f"  {k_val}: Mean T2 = {metrics['mean_T2']:.4f}, Mean Q = {metrics['mean_Q']:.4f}")

# Compute metrics for shifted mean data
shifted_mean_metrics = diagnostic_system.compute_metrics(new_realizations_shifted_mean)
print("\nMetrics for Shifted Mean New Realizations:")
for k_val, metrics in shifted_mean_metrics.items():
    print(f"  {k_val}: Mean T2 = {metrics['mean_T2']:.4f}, Mean Q = {metrics['mean_Q']:.4f}")

# Compute metrics for increased variance data
increased_variance_metrics = diagnostic_system.compute_metrics(new_realizations_increased_variance)
print("\nMetrics for Increased Variance New Realizations:")
for k_val, metrics in increased_variance_metrics.items():
    print(f"  {k_val}: Mean T2 = {metrics['mean_T2']:.4f}, Mean Q = {metrics['mean_Q']:.4f}")

Baseline archive shape: (3000, 4)
New realizations (nominal) shape: (100, 4)

Metrics for Nominal New Realizations:
  k=1: Mean T2 = 1.0482, Mean Q = 2.5223
  k=2: Mean T2 = 2.0110, Mean Q = 1.2523
  k=3: Mean T2 = 2.9509, Mean Q = 0.4683

Metrics for Shifted Mean New Realizations:
  k=1: Mean T2 = 1.4398, Mean Q = 6.4125
  k=2: Mean T2 = 2.9880, Mean Q = 4.3701
  k=3: Mean T2 = 4.0765, Mean Q = 3.4622

Metrics for Increased Variance New Realizations:
  k=1: Mean T2 = 1.7764, Mean Q = 6.5205
  k=2: Mean T2 = 4.4939, Mean Q = 2.9357
  k=3: Mean T2 = 6.4260, Mean Q = 1.3242
